In [3]:
# 1. Xử lý dữ liệu và hệ thống
import os
import glob
import joblib
import numpy as np
import pandas as pd
import kagglehub

# 2. Trực quan hóa dữ liệu
import seaborn as sns
import matplotlib.pyplot as plt

# 3. Tiền xử lý và Lựa chọn đặc trưng (Preprocessing & Feature Selection)
from sklearn.preprocessing import (
    StandardScaler, 
    RobustScaler, 
    LabelEncoder, 
    QuantileTransformer
)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.kernel_approximation import Nystroem
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

# 4. Chia dữ liệu và Mô hình (Model Selection & Algorithms)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import BaggingClassifier

# 5. Đánh giá mô hình (Evaluation Metrics)
from sklearn.metrics import (
    classification_report, 
    confusion_matrix
)

In [ ]:
OUTPUT_DIR = 'data/artifacts'
# Download latest version
path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")

print("Path to dataset files:", path)

# Danh sách 17 features bạn yêu cầu
selected_features = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean',
    'Packet Length Std', 'SYN Flag Count', 'ACK Flag Count',
    'FIN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'URG Flag Count'
]

In [ ]:
files = os.listdir(path)
print("Các file trong dataset:", files)

# Thông thường dataset này gồm nhiều file CSV, ta sẽ đọc một file hoặc gộp chúng lại
# Giả sử ta đọc file đầu tiên trong danh sách (hoặc file có tên cụ thể)
csv_files = [f for f in files if f.endswith('.csv')]
print("Danh sách file CSV:", csv_files)

In [ ]:
# 1. Đọc file CSV (Ví dụ lấy file đầu tiên)

df_list = []
print(f"--- Đang đọc và lọc {len(csv_files)} files ---")
    
for file in csv_files:
    temp_df = pd.read_csv(path + '/' + file)
    temp_df.columns = temp_df.columns.str.strip()
        
    cols_to_keep = selected_features + ['Label']
    cols_present = [col for col in cols_to_keep if col in temp_df.columns]
        
    df_list.append(temp_df[cols_present])

df = pd.concat(df_list, ignore_index=True)
#df = pd.read_csv(file_path)

df.columns = df.columns.str.strip()

target_column = 'Label' 
all_needed_columns = selected_features + [target_column]

# Lọc lấy các cột cần thiết
df_final = df[all_needed_columns].copy()

df_final.replace([np.inf, -np.inf], np.nan, inplace=True)
df_final.dropna(inplace=True)

print("Kích thước dữ liệu sau khi lọc:", df_final.shape)
df_final.head()

In [ ]:
print("--- KÍCH THƯỚC DỮ LIỆU ---")
print(f"Số dòng: {df.shape[0]}, Số cột: {df.shape[1]}")

print("\n--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---")
display(df.head()) # Dùng display() trong Kaggle/Jupyter sẽ hiển thị bảng đẹp hơn print()

print("\n--- THỐNG KÊ KIỂU DỮ LIỆU & GIÁ TRỊ THIẾU ---")
# Lấy danh sách các cột có chứa giá trị NaN (nếu có)
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])
# 3. Chuẩn hóa tên (Phòng trường hợp cột tên là 'label' viết thường hoặc 'Class')
if 'label' in df.columns:
    df.rename(columns={'label': 'Label'}, inplace=True)
elif 'Class' in df.columns:
    df.rename(columns={'Class': 'Label'}, inplace=True)

# 2. Làm sạch nhãn (Dùng Regex để sửa lỗi ký tự lạ)
print("--- Đang dọn dẹp lỗi Encoding của nhãn ---")
if 'Label' in df.columns:
    df['Label'] = df['Label'].replace(to_replace=r'.*Brute Force.*', value='Web Attack - Brute Force', regex=True)
    df['Label'] = df['Label'].replace(to_replace=r'.*XSS.*', value='Web Attack - XSS', regex=True)
    df['Label'] = df['Label'].replace(to_replace=r'.*Sql Injection.*', value='Web Attack - Sql Injection', regex=True)
        
    df.dropna(subset=['Label'], inplace=True)
    
print("\n--- PHÂN BỔ CÁC LỚP NHÃN (LABEL) ---")
if 'Label' in df.columns:
    print(df['Label'].value_counts())
else:
    print("Không tìm thấy cột 'Label', vui lòng kiểm tra lại tên cột.")

In [ ]:

# Tách Features và Label
X = df_final[selected_features]
y = df_final[target_column]

# Chuẩn hóa dữ liệu (SVM cực kỳ nhạy cảm với thang đo, bắt buộc phải Scale)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


print("Sẵn sàng cho bước Undersampling và huấn luyện SVM!")

In [ ]:

def svm_quantile_optimization_final(df, selected_features):
    print("\n" + "="*60)
    print("HỆ THỐNG SVM TỐI ƯU (QUANTILE + NYSTROEM) - EXPORT VERSION")
    print("="*60)

    # 1. Dọn dẹp dữ liệu cực kỳ kỹ lưỡng
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(subset=selected_features + ['Label'], inplace=True)
    
    # Khống chế Outliers bằng Clipping
    for col in selected_features:
        upper_limit = df[col].quantile(0.99)
        df[col] = np.clip(df[col], a_min=None, a_max=upper_limit)
    
    print("--- 1. Đã dọn dẹp Inf/NaN và Clipping xong ---")

    # 2. Lấy mẫu dữ liệu để cân bằng tốc độ huấn luyện
    df_attacks = df[df['Label'] != 'BENIGN']
    df_benign = df[df['Label'] == 'BENIGN'].sample(n=100000, random_state=42)
    df_sampled = pd.concat([df_benign, df_attacks])

    X = df_sampled[selected_features]
    le = LabelEncoder()
    y = le.fit_transform(df_sampled['Label'])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. Xây dựng Pipeline Chiến thuật Quantile (600 components)
    svm_pipeline = Pipeline([
        ('scaler', QuantileTransformer(output_distribution='normal', random_state=42)),
        ('feature_map', Nystroem(kernel='rbf', gamma=0.1, n_components=600, random_state=42)),
        ('svm', LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, dual=False))
    ])

    print("--- 2. Đang huấn luyện mô hình (Quá trình này có thể mất vài phút)... ---")
    svm_pipeline.fit(X_train, y_train)

    # 4. Dự đoán và Báo cáo
    y_pred = svm_pipeline.predict(X_test)
    print("\n" + "="*60)
    print("BÁO CÁO KẾT QUẢ SVM CUỐI CÙNG")
    print("="*60)
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

    # 5. LƯU TRỮ ARTIFACTS (PKL & PNG)
    export_dir = "data/modules"
    os.makedirs(export_dir, exist_ok=True)
    
    # Lưu Pipeline (bao gồm cả QuantileTransformer, Nystroem và SVM)
    model_path = os.path.join(export_dir, 'svm_final_v5_model.pkl')
    joblib.dump(svm_pipeline, model_path)
    
    # Lưu LabelEncoder để giải mã nhãn sau này
    encoder_path = os.path.join(export_dir, 'svm_label_encoder_v5.pkl')
    joblib.dump(le, encoder_path)
    
    print(f"--- 3. Đã lưu Model và Encoder tại: {export_dir} ---")

    # 6. VẼ VÀ LƯU MA TRẬN NHẦM LẪN
    print("--- 4. Đang tạo ma trận nhầm lẫn ---")
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', 
                xticklabels=le.classes_, yticklabels=le.classes_)
    
    plt.title('Confusion Matrix - SVM Quantile Optimization (V5)', fontsize=14)
    plt.ylabel('Thực tế (True Label)')
    plt.xlabel('Dự đoán (Predicted Label)')
    plt.tight_layout()

    cm_path = os.path.join(OUTPUT_DIR, "svm_v5_confusion_matrix.png")
    plt.savefig(cm_path)
    print(f"--- 5. Đã lưu Ma trận nhầm lẫn tại: {cm_path} ---")
    plt.show()

# Thực thi hàm cuối cùng
svm_quantile_optimization_final(df, selected_features)